In [ ]:
# This script is the implementation of random splitting and early stopping where just one set of hyperparameters is tested.
# It also implements the weighed cross entropy loss.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import os
import pickle as pkl
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from model import LSTM
# from comet_ml import Experiment

# Load data
DATASETS_PATH = os.path.join('..','..','..','data')
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, 'test.pickle')
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, 'train.pickle')

with open(TEST_DATASET_PATH, 'rb') as f:
    test_dataset = pkl.load(f)
with open(TRAIN_DATASET_PATH, 'rb') as f:
    train_dataset = pkl.load(f)

# Dataset class
class SensorDataSet(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, idx):
        data = self.dataset.iloc[idx]
        x = torch.tensor(data['sensor_data'], dtype=torch.float32)
        y = torch.tensor(data['label'], dtype=torch.long)
        return x, y

    def __len__(self):
        return len(self.dataset)

# Evaluate function
def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = loss_fn(output, y)
            total_loss += loss.item() * x.size(0)
            pred = torch.argmax(output, dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

# Train function
def train_model(model, train_loader, val_loader, loss_fn, optimizer, device, max_epochs=20, patience=3, experiment=None):
    best_val_acc = 0
    patience_counter = 0
    for epoch in range(max_epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)
            loss = loss_fn(output, y)
            loss.backward()
            optimizer.step()

        # Evaluate on validation set
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

        # Comet logging
        if experiment:
            experiment.log_metric("val_accuracy", val_acc, step=epoch)
            experiment.log_metric("val_loss", val_loss, step=epoch)

        # === Show predictions for the first batch ===
        model.eval()
        with torch.no_grad():
            for val_x, val_y in val_loader:
                val_x, val_y = val_x.to(device), val_y.to(device)
                pred_logits = model(val_x)
                pred_classes = torch.argmax(pred_logits, dim=1)
                print(f"[Epoch {epoch+1}] Predictions: {pred_classes[:10].cpu().numpy()} | Labels: {val_y[:10].cpu().numpy()}")
                break 
        # ===========================

        # Checking if the model improved on validation set
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break # Stop training since model is not improving further
    # Load best state
    model.load_state_dict(best_model_state) 
    return best_val_acc

# HYPERPARAMETERS
input_dim = 6
num_classes = 12
hidden_dim = 256
num_layers = 2
dropout = 0
batch_size = 32
learning_rate = 0.0001
weight_decay = 0.0001
max_epochs = 50
patience = 5

# Log experiment name and parameters
# project_name = f"LSTM_hd{hidden_dim}_nl{num_layers}_do{dropout}_bs{batch_size}_lr{learning_rate}_wd{weight_decay}"

# 10 random states for splits
split_random_states = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_accuracies = []
test_loader = DataLoader(SensorDataSet(test_dataset), batch_size=batch_size, shuffle=False)

# Training and evaluating on each random split
for run_idx, seed in enumerate(split_random_states):
    # experiment = Experiment(
    #     api_key="",
    #     project_name=project_name,
    #     auto_output_logging=False,
    #     auto_metric_logging=False,
    #     auto_param_logging=False
    # )
    # experiment.set_name(f"split_{run_idx+1}")
    # experiment.log_parameters({
    #     "hidden_dim": hidden_dim,
    #     "num_layers": num_layers,
    #     "dropout": dropout,
    #     "batch_size": batch_size,
    #     "learning_rate": learning_rate,
    #     "weight_decay": weight_decay,
    #     "input_dim": input_dim,
    #     "num_classes": num_classes,
    #     "max_epochs": max_epochs,
    #     "patience": patience, 
    #     "split_seed": seed
    # })

    # Perform the random split
    train_df, val_df = train_test_split(
        train_dataset,
        test_size=0.2,
        stratify=train_dataset['label'],
        random_state=seed
    )

    # DataLoaders
    train_loader = DataLoader(SensorDataSet(train_df), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SensorDataSet(val_df), batch_size=batch_size, shuffle=False)

    # Model
    model = LSTM(input_dim, hidden_dim, num_classes, num_layers, dropout).to(device)

    # Weighed cross entropy loss 
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_df['label']),
        y=train_df['label']
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor) # if you do not want the weighed loss just delete the argument in this line

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Training and testing
    _ = train_model(model, train_loader, val_loader, loss_fn, optimizer, device) # if you wanna use comet: add experiment as an additional argument
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)

    # experiment.log_metric("test_accuracy", test_acc)
    # experiment.log_metric("test_loss", test_loss)

    test_accuracies.append(test_acc)
    # experiment.end()

# Resulting metrics
test_accuracies = np.array(test_accuracies)
mean_val = test_accuracies.mean()
std_val = test_accuracies.std()

# Summary experiment logs the averages fromt he random splits above and the results for one final training on the full dataset without a validation split
# summary_experiment = Experiment(
#     api_key="",
#     project_name=project_name,
#     auto_output_logging=False,
#     auto_metric_logging=False,
#     auto_param_logging=False
# )
# summary_experiment.set_name("summary")

# summary_experiment.log_metric("test_mean_accuracy", mean_val)
# summary_experiment.log_metric("test_std_accuracy", std_val)

print("=== Cross-Validation Summary ===")
print(f"Mean Accuracy: {mean_val:.4f}")
print(f"Std Accuracy: {std_val:.4f}")

# Final training on whole train dataset
train_loader = DataLoader(SensorDataSet(train_df), batch_size=batch_size, shuffle=True)

# Model
final_model = LSTM(input_dim, hidden_dim, num_classes, num_layers, dropout).to(device)

# Weighed cross entropy loss 
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_loader['label']),
    y=train_df['label']
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor) # if you do not want the weighed loss just delete the argument in this line

#Optimizer
optimizer = torch.optim.Adam(final_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Training and testing
_ = train_model(final_model, train_loader, val_loader, loss_fn, optimizer, device) # if you wanna use comet: add experiment as an additional argument
test_loss, test_acc = evaluate(final_model, test_loader, loss_fn, device)

# summary_experiment.log_metric("test_accuracy", test_acc)
# summary_experiment.log_metric("test_loss", test_loss)
# summary_experiment.end()

print("=== Final Test Evaluation ===")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")